# Data cleaning

## Loading libraries and files

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path  

pd.set_option("display.max_columns", None)  
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

In [3]:
beds = pd.read_csv(RAW_DIR/"eurostat_hospital_beds.csv")
discharges = pd.read_csv(RAW_DIR/"eurostat_hospital_discharges.csv")
length_stay = pd.read_csv(RAW_DIR/"eurostat_average_length_stay.csv")

## Defining the scope

In [ ]:
COUNTRIES = ["ES", "PT", "FR", "IT", "DE"]
ANALYSIS_YEARS = list(range(2014,2020))
DIAGNOSES = ["A-T_Z", "C00-D48", "F", "I", "J"]

## Creating filtered versions of each dataframe

In [11]:
beds_scoped = (
    beds.loc[beds["geo"].isin(COUNTRIES) & beds["TIME_PERIOD"].isin(ANALYSIS_YEARS)]
    .copy()
)

In [12]:
discharges_scoped = (
    discharges.loc[
        discharges["geo"].isin(COUNTRIES)
        & discharges["TIME_PERIOD"].isin(ANALYSIS_YEARS)
        & discharges["icd10"].isin(DIAGNOSES)
    ]
    .copy()
)

In [13]:
length_stay_scoped = (
    length_stay.loc[
        length_stay["geo"].isin(COUNTRIES)
        & length_stay["TIME_PERIOD"].isin(ANALYSIS_YEARS)
        & length_stay["icd10"].isin(DIAGNOSES)
    ]
    .copy()
)

In [ ]:
print("Beds:", beds_scoped.shape)
print("Discharges:", discharges_scoped.shape)
print("Average length of stay:", length_stay_scoped.shape)

Beds: (30, 19)
Discharges: (138, 25)
Average length of stay: (139, 25)


## Checking for unexpected categories

In [23]:
for name, df in {
    "beds": beds_scoped,
    "discharges": discharges_scoped,
    "length_stay": length_stay_scoped
}.items():
    print(f"\n{name.upper()}")
    print("Countries:", sorted(df["geo"].unique()))
    print("Years:", sorted(df["TIME_PERIOD"].unique()))
    
    if "icd10" in df.columns:
        print("Diagnoses:", sorted(df["icd10"].unique()))


BEDS
Countries: ['DE', 'ES', 'FR', 'IT', 'PT']
Years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]

DISCHARGES
Countries: ['DE', 'ES', 'FR', 'IT', 'PT']
Years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Diagnoses: ['A-T_Z', 'C00-D48', 'F', 'I', 'J']

LENGTH_STAY
Countries: ['DE', 'ES', 'FR', 'IT', 'PT']
Years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Diagnoses: ['A-T_Z', 'C00-D48', 'F', 'I', 'J']


## Validating the constant filters already applied when downloading

In [ ]:
print(beds_scoped[["facility", "unit"]].drop_duplicates())
print(discharges_scoped[["age", "sex", "indic_he", "unit"]].drop_duplicates())
print(length_stay_scoped[["age", "sex", "indic_he", "unit"]].drop_duplicates())

# HBEDT = Available beds in hospitals 
# P_HTHAB = Per hundred thousand inhabitants
# INPAT = in-patients (total number)
# ALOS = In-patient average length of stay (in days)
# NR = number (days)

  facility     unit
0    HBEDT  P_HTHAB
     age sex indic_he     unit
0  TOTAL   T    INPAT  P_HTHAB
     age sex indic_he unit
0  TOTAL   T     ALOS   NR


All datasets were restricted to Germany, Spain, France, Italy and Portugal, covering 2014-2019. Hospital discharges and average lenght of stay were further restricted to the five selected ICD-10 diagnosis groups. Constant dimensions selected during extraction were validated and retainted only as source metadata until the transformation stage. 

## Selection and standardisation of analytical fields

The Eurostat extracts are already stored in long format. Only fields required for the analytical model, source traceability and data-quality checks are retained.

In [ ]:
beds_tidy = (
    beds_scoped[["geo", "TIME_PERIOD", "OBS_VALUE", "unit", "OBS_FLAG"]]
    .rename(
        columns={
            "geo": "country_code",
            "TIME_PERIOD": "year",
            "OBS_VALUE": "value",
            "unit": "unit_code",
            "OBS_FLAG": "status_flag"
        }
    )
    .copy()
)

In [56]:
discharges_tidy = (
    discharges_scoped[["geo", "TIME_PERIOD", "icd10", "OBS_VALUE", "unit", "OBS_FLAG"]]
    .rename(
        columns={
            "geo": "country_code",
            "TIME_PERIOD": "year",
            "icd10": "diagnosis_code",
            "OBS_VALUE": "value",
            "unit": "unit_code",
            "OBS_FLAG": "status_flag"
        }
    )
    .copy()
)

In [57]:
length_stay_tidy = (
    length_stay_scoped[["geo", "TIME_PERIOD", "icd10", "OBS_VALUE", "unit", "OBS_FLAG"]]
    .rename(
        columns={
            "geo": "country_code",
            "TIME_PERIOD": "year",
            "icd10": "diagnosis_code",
            "OBS_VALUE": "value",
            "unit": "unit_code",
            "OBS_FLAG": "status_flag"
        }
    )
    .copy()
)

In [ ]:
for name, df in {
    "beds": beds_tidy,
    "discharges": discharges_tidy,
    "length_stay": length_stay_tidy
}.items():
    print(f"\n{name.upper()}")
    print(df.shape)
    print(df.head(3))
    print(df.dtypes)


BEDS
(30, 5)
  country_code  year   value unit_code status_flag
0           DE  2014  822.82   P_HTHAB         NaN
1           DE  2015  813.31   P_HTHAB         NaN
2           DE  2016  806.26   P_HTHAB         NaN
country_code        str
year              int64
value           float64
unit_code           str
status_flag         str
dtype: object

DISCHARGES
(138, 6)
  country_code  year diagnosis_code    value unit_code  status_flag
0           DE  2014          A-T_Z  25602.5   P_HTHAB          NaN
1           DE  2015          A-T_Z  25534.3   P_HTHAB          NaN
2           DE  2016          A-T_Z  25685.6   P_HTHAB          NaN
country_code          str
year                int64
diagnosis_code        str
value             float64
unit_code             str
status_flag       float64
dtype: object

LENGTH_STAY
(139, 6)
  country_code  year diagnosis_code  value unit_code  status_flag
0           DE  2014          A-T_Z    9.0        NR          NaN
1           DE  2015          A

## Missing observations

Expected granularity for each dataframe: 
- beds: 5 countries x 6 years = 30 → 30 rows
- discharges: 5 countries x 6 years x 5 diagnosis = 150 → 138 rows → 12 missing
- lenght_stay: 5 countries x 6 years x 5 diagnosis = 150 → 139 rows → 11 missing

Missing combinations represent observations that are not published in the Eurostat source for the selected country, year and diagnosis group. They do not represent zero values and are not imputed. 

Analysis requiring a complete country-year-diagnosis panel will exclude unavailable combinations, while descriptive analysis will clearly indicate incomplete coverage where relevant. 

In [61]:
def find_missing_combinations(df, countries, years, diagnoses=None):
    """
    Return expected combinations that are absent from a dataframe
    """
    if diagnoses is None:
        expected = (
            pd.MultiIndex.from_product(
                [countries, years],
                names=["country_code", "year"]
            )
            .to_frame(index=False)
        )
        
        keys = ["country_code", "year"]
    
    else:
        expected = (
            pd.MultiIndex.from_product(
                [countries, years, diagnoses],
                names=["country_code", "year", "diagnosis_code"]
            )
            .to_frame(index=False)
        )
        
        keys = ["country_code", "year", "diagnosis_code"]
    
    missing = (
        expected.merge(
            df[keys].drop_duplicates(),
            on=keys,
            how="left",
            indicator=True
        )
        .query("_merge == 'left_only'")
        .drop(columns="_merge")
    )
    
    return missing

In [62]:
missing_beds = find_missing_combinations(
    beds_tidy,
    COUNTRIES,
    ANALYSIS_YEARS
)

missing_discharges = find_missing_combinations(
    discharges_tidy,
    COUNTRIES,
    ANALYSIS_YEARS,
    DIAGNOSES
)

missing_length_stay = find_missing_combinations(
    length_stay_tidy,
    COUNTRIES,
    ANALYSIS_YEARS,
    DIAGNOSES
)

In [63]:
print("Missing beds combinations:", len(missing_beds))
print("Missing discharges combinations:", len(missing_discharges))
print("Missing length-stay combinations:", len(missing_length_stay))

display(missing_beds)
display(missing_discharges)
display(missing_length_stay)

Missing beds combinations: 0
Missing discharges combinations: 12
Missing length-stay combinations: 11


,country_code,year


,country_code,year,diagnosis_code
0,ES,2014,A-T_Z
40,PT,2016,A-T_Z
41,PT,2016,C00-D48
42,PT,2016,F
43,PT,2016,I
44,PT,2016,J
45,PT,2017,A-T_Z
46,PT,2017,C00-D48
47,PT,2017,F
48,PT,2017,I


,country_code,year,diagnosis_code
0,ES,2014,A-T_Z
40,PT,2016,A-T_Z
41,PT,2016,C00-D48
42,PT,2016,F
43,PT,2016,I
44,PT,2016,J
45,PT,2017,A-T_Z
46,PT,2017,C00-D48
47,PT,2017,F
48,PT,2017,I


## Data-type validation and status-flag handling

In [69]:
beds_clean.columns

Index(['country_code', 'year', 'value', 'unit_code', 'status_flag'], dtype='str')

In [73]:
def standarise_data_types(df, has_diagnosis=False):
    """
    Standarise data types and text fields for a processed analytical dataset
    """

    result = df.copy()

    result["country_code"] = (result["country_code"].astype("string").str.strip())
    result["year"] = result["year"].astype("int64")
    result["value"] = result["value"].astype("float64")
    result["unit_code"] = (result["unit_code"].astype("string").str.strip().replace("",pd.NA))
    result["status_flag"] = (result["status_flag"].astype("string").str.strip().replace("",pd.NA).astype("string"))

    if has_diagnosis:
        result["diagnosis_code"] = (result["diagnosis_code"].astype("string").str.strip())
    
    return (result)

In [74]:
beds_clean = standarise_data_types(beds_tidy)
discharges_clean = standarise_data_types(discharges_tidy, has_diagnosis=True)
length_stay_clean = standarise_data_types(length_stay_tidy, has_diagnosis=True)

In [75]:
for name, df, keys in [
    ("beds", beds_clean, ["country_code", "year"]),
    ("discharges", discharges_clean, ["country_code", "year", "diagnosis_code"]),
    ("length_stay", length_stay_clean, ["country_code", "year", "diagnosis_code"])
]:

    print(f"\n{name.upper()}")
    print("Data types:")
    print(df.dtypes)
    
    print("\nNull values:")
    print(df.isna().sum())

    print("\nDuplicate analytical keys:")
    print(df.duplicated(subset=keys).sum())


BEDS
Data types:
country_code     string
year              int64
value           float64
unit_code        string
status_flag      string
dtype: object

Null values:
country_code     0
year             0
value            0
unit_code        0
status_flag     29
dtype: int64

Duplicate analytical keys:
0

DISCHARGES
Data types:
country_code       string
year                int64
diagnosis_code     string
value             float64
unit_code          string
status_flag        string
dtype: object

Null values:
country_code        0
year                0
diagnosis_code      0
value               0
unit_code           0
status_flag       138
dtype: int64

Duplicate analytical keys:
0

LENGTH_STAY
Data types:
country_code       string
year                int64
diagnosis_code     string
value             float64
unit_code          string
status_flag        string
dtype: object

Null values:
country_code        0
year                0
diagnosis_code      0
value               0
unit_code       


All analytical keys are unique and missing values are limited to `status_flag`, where a null value indicates that Eurostat did not report an observation-status flag. No duplicates present. 

## Dimension tables

Dimension tables provide consistent descriptive attributes for countries, years and diagnosis groups. They are separated from the fact tables to avoid repeating labels and to create stable relationships in MySQL and Power BI. 


In [76]:
dim_country = pd.DataFrame({
    "country_id": [1,2,3,4,5],
    "country_code": ["DE", "ES", "FR", "IT", "PT"],
    "country_name": ["Germany", "Spain", "France", "Italy", "Portugal"]
})

dim_year = pd.DataFrame({
    "year_id": ANALYSIS_YEARS,
    "year": ANALYSIS_YEARS
})

dim_diagnosis = pd.DataFrame({
    "diagnosis_id": [1,2,3,4,5],
    "diagnosis_code": ["A-T_Z", "C00-D48", "F", "I", "J"],
    "diagnosis_name": [
        "All diseases", 
        "Neoplasms", 
        "Mental and behavioural disorders", 
        "Diseases of the circulatory sistem", 
        "Diseases of the respiratory system"], 
    "diagnosis_group": [
        "Total",
        "Neoplasms",
        "Mental health",
        "Circulatory",
        "Respiratory"
    ]
}
)

In [77]:
for name, df in {
    "dim_country": dim_country,
    "dim_year": dim_year,
    "dim_diagnosis": dim_diagnosis
}.items():
    print(f"\n{name.upper()} - {df.shape[0]} rows")
    display(df)
    print("Duplicate primary keys:", df.iloc[:,0].duplicated().sum())


DIM_COUNTRY - 5 rows


,country_id,country_code,country_name
0,1,DE,Germany
1,2,ES,Spain
2,3,FR,France
3,4,IT,Italy
4,5,PT,Portugal


Duplicate primary keys: 0

DIM_YEAR - 6 rows


,year_id,year
0,2014,2014
1,2015,2015
2,2016,2016
3,2017,2017
4,2018,2018
5,2019,2019


Duplicate primary keys: 0

DIM_DIAGNOSIS - 5 rows


,diagnosis_id,diagnosis_code,diagnosis_name,diagnosis_group
0,1,A-T_Z,All diseases,Total
1,2,C00-D48,Neoplasms,Neoplasms
2,3,F,Mental and behavioural disorders,Mental health
3,4,I,Diseases of the circulatory sistem,Circulatory
4,5,J,Diseases of the respiratory system,Respiratory


Duplicate primary keys: 0


## Fact Tables

Fact tables store the numerical observations and reference the dimension tables through foreign keys. Natural codes and descriptive labels are replaced with surrogate keys to create a star-schema-ready data model.

In [ ]:
def add_dimension_keys(df, has_diagnosis=False):
    """
    Add dimension IDs to an analytical dataframe using validated manu-to-one joins.
    """

    result = (
        df.merge(
            dim_country,
            on="country_code",
            how="left",
            validate="many_to_one"
        )
        .merge(
            dim_year,
            on="year",
            how="left",
            validate="many_to_one"
        )
    )

    if has_diagnosis:
        result = result.merge(
            dim_diagnosis,
            on="diagnosis_code",
            how="left",
            validate="many_to_one"
        )
    
    return result

    # validate="many_to_one" obliga a que cada código de país, año o diagnóstico encuentre una sola fila en su dimensión
    # Si creases una dimensión con duplicados, el código se detendría en vez de duplicar silenciosamente las filas de hechos.

In [82]:
fact_beds = (
    add_dimension_keys(beds_clean)
    [["country_id", "year_id", "value", "unit_code", "status_flag"]].copy()
)

In [ ]:
fact_discharges = (
    add_dimension_keys(discharges_clean, has_diagnosis=True)
    [["country_id", "year_id", "diagnosis_id", "value", "unit_code", "status_flag"]]
    .copy()
)

In [84]:
fact_length_stay = (
    add_dimension_keys(length_stay_clean, has_diagnosis=True)
    [["country_id", "year_id", "diagnosis_id", "value", "unit_code", "status_flag"]]
    .copy()
)

In [85]:
fact_tables = {
    "fact_beds":(fact_beds,["country_id", "year_id"]),
    "fact_discharges":(fact_discharges,["country_id", "year_id", "diagnosis_id"]),
    "fact_length_stay":(fact_length_stay,["country_id", "year_id", "diagnosis_id"])
}

for name, (df,keys) in fact_tables.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print("Missing foreing keys:", df[keys].isna().sum().sum())
    print("Duplicate anaytical keys:", df.duplicated(subset=keys).sum())
    display(df.head(3))


FACT_BEDS
Shape: (30, 5)
Missing foreing keys: 0
Duplicate anaytical keys: 0


,country_id,year_id,value,unit_code,status_flag
0,1,2014,822.82,P_HTHAB,<NA>
1,1,2015,813.31,P_HTHAB,<NA>
2,1,2016,806.26,P_HTHAB,<NA>



FACT_DISCHARGES
Shape: (138, 6)
Missing foreing keys: 0
Duplicate anaytical keys: 0


,country_id,year_id,diagnosis_id,value,unit_code,status_flag
0,1,2014,1,25602.5,P_HTHAB,<NA>
1,1,2015,1,25534.3,P_HTHAB,<NA>
2,1,2016,1,25685.6,P_HTHAB,<NA>



FACT_LENGTH_STAY
Shape: (139, 6)
Missing foreing keys: 0
Duplicate anaytical keys: 0


,country_id,year_id,diagnosis_id,value,unit_code,status_flag
0,1,2014,1,9.0,NR,<NA>
1,1,2015,1,9.0,NR,<NA>
2,1,2016,1,8.9,NR,<NA>


## Export and reload validation
Processed dimension and fact tables are exported as csv files. Each file is then reloaded and checked to verify that row counts, column names and values are preserved during export. 

In [87]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

processed_tables = {
    "dim_country": dim_country,
    "dim_year": dim_year,
    "dim_diagnosis": dim_diagnosis,
    "fact_beds": fact_beds,
    "fact_discharges": fact_discharges,
    "fact_length_stay": fact_length_stay
}

for table_name, df in processed_tables.items():
    output_path = PROCESSED_DIR / f"{table_name}.csv"

    df.to_csv(output_path, index=False, encoding="utf-8")

    print(f"Saved: {output_path}")

Saved: ..\data\processed\dim_country.csv
Saved: ..\data\processed\dim_year.csv
Saved: ..\data\processed\dim_diagnosis.csv
Saved: ..\data\processed\fact_beds.csv
Saved: ..\data\processed\fact_discharges.csv
Saved: ..\data\processed\fact_length_stay.csv


In [ ]:
reloaded_tables = {}

for table_name, original_df in processed_tables.items():
    input_path = PROCESSED_DIR / f"{table_name}.csv"

    reloaded_df = pd.read_csv(input_path, dtype={"status_flag":"string"})

    reloaded_tables[table_name] = reloaded_df

    pd.testing.assert_frame_equal(
        original_df.reset_index(drop=True),
        reloaded_df.reset_index(drop=True),
        check_dtype=False
    )

    print(
        f"{table_name}: "
        f"{reloaded_df.shape[0]} rows, "
        f"{reloaded_df.shape[1]} columns — validation passed"
    )

dim_country: 5 rows, 3 columns — validation passed
dim_year: 6 rows, 2 columns — validation passed
dim_diagnosis: 5 rows, 4 columns — validation passed
fact_beds: 30 rows, 5 columns — validation passed
fact_discharges: 138 rows, 6 columns — validation passed
fact_length_stay: 139 rows, 6 columns — validation passed
